In [1]:
import numpy as np
import cv2 as cv
from ultralytics import YOLO
import time
import pycaw.pycaw as pycaw
from ctypes import cast, POINTER
from comtypes import CLSCTX_ALL

from pathlib import Path

In [2]:
pr_dir = Path.cwd().parents[0]
path_model = pr_dir / 'models' / 'yolo11n.pt'
model = YOLO(path_model)

In [ ]:
def gesture_old(recording = False):
    cap = cv.VideoCapture(0)

    if not cap.isOpened():
        print('Камера недоступна')
        return
    

    if recording:
        fourcc = cv.VideoWriter_fourcc(*'XVID')
        fps = 20.0
        width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))

        out = cv.VideoWriter('output.avi', fourcc, fps, (width, height))

    ret, frame = cap.read()

    model = YOLO("yolo11n.pt")

    time_t = time.time()

    frame_after = frame.copy()

    while cap.isOpened():
        ret, frame = cap.read()
        #frame = cv.resize(frame, (640, 480))

        if not ret:
            break
        if cv.waitKey(1) & 0xFF == ord('q'):
            break
        
        if time.time() - time_t >= 1:
            print(1)
            time_t = time.time()
        
            results = model(frame)
            frame_after = frame.copy()
            for i,result in enumerate (results):
                xyxy = result.boxes.xyxy
                for ix,iy,x,y in xyxy: 
                    ix = int(ix)
                    iy = int(iy)
                    x = int(x)
                    y = int(y)
                    cv.rectangle(frame_after,(ix,iy),(x,y),(0,255,0),5)
        
        
        cv.imshow('frame_after', frame_after)
        cv.imshow('frame', frame)
        
        
        if recording:
            out.write(frame)
    cap.release()
    if recording:
            out.release()
            
    cv.destroyAllWindows()


In [5]:
def gesture(recording = False):
    cap = cv.VideoCapture(0)

    if not cap.isOpened():
        print('Камера недоступна')
        return
    

    if recording:
        fourcc = cv.VideoWriter_fourcc(*'XVID')
        fps = 20.0
        width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))

        out = cv.VideoWriter('output.avi', fourcc, fps, (width, height))

    ret, frame = cap.read()

    model = YOLO("yolo11n-pose.pt")

    time_t = time.time()

    frame_after = frame.copy()

    while cap.isOpened():
        ret, frame = cap.read()
        frame = cv.resize(frame, (640, 480))

        if not ret:
            break
        if cv.waitKey(1) & 0xFF == ord('q'):
            break
        
        if time.time() - time_t >= 1:
            print(1)
            time_t = time.time()
        
            results = model(frame)
            frame_after = frame.copy()
            for i,result in enumerate (results):
                xy = result.keypoints.xy
                for x,y in xy[0]:
                    #print(x,y)
                    cv.circle(frame_after, (int(x),int(y)), 1, (255, 0, 0), -1)
        
        
            cv.imshow('frame_after', frame_after)
        cv.imshow('frame', frame)
        
        
        if recording:
            out.write(frame)
    cap.release()
    if recording:
            out.release()
            
    cv.destroyAllWindows()


In [ ]:
def set_system_volume(level):
    """Установить уровень громкости (0.0 - 1.0)"""
    devices = pycaw.AudioUtilities.GetSpeakers()
    interface = devices.Activate(
        pycaw.IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
    volume = cast(interface, POINTER(pycaw.IAudioEndpointVolume))
    volume.SetMasterVolumeLevelScalar(level, None)

def get_system_volume():
    """Получить текущий уровень громкости"""
    devices = pycaw.AudioUtilities.GetSpeakers()
    interface = devices.Activate(
        pycaw.IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
    volume = cast(interface, POINTER(pycaw.IAudioEndpointVolume))
    return volume.GetMasterVolumeLevelScalar()

In [6]:
gesture(recording=False)

1

0: 480x640 1 person, 155.7ms
Speed: 3.2ms preprocess, 155.7ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
1

0: 480x640 1 person, 143.3ms
Speed: 3.9ms preprocess, 143.3ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)
1

0: 480x640 1 person, 231.2ms
Speed: 3.8ms preprocess, 231.2ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)
1

0: 480x640 1 person, 121.9ms
Speed: 2.4ms preprocess, 121.9ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)
1

0: 480x640 1 person, 160.1ms
Speed: 3.6ms preprocess, 160.1ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)
1

0: 480x640 1 person, 167.9ms
Speed: 2.8ms preprocess, 167.9ms inference, 2.6ms postprocess per image at shape (1, 3, 480, 640)
1

0: 480x640 1 person, 164.0ms
Speed: 4.3ms preprocess, 164.0ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
1

0: 480x640 1 person, 176.0ms
Speed: 3.0ms preprocess, 176.0ms inference, 1.6ms postproc

In [39]:
path_img = pr_dir / 'data' / 'test.png'
image = cv.imread(path_img.as_posix())
results = model(
           image, 
           stream = False, 
           #vid_stride = 25,
           verbose = False)
if len(results) == 0: print('Объектов нет')
else: 
    for result in results:
       # result.show()
        xy = result.boxes.xyxy
        names = [result.names[cls.item()] for cls in results[0].boxes.cls.int()]
        #print(names,xy,xy.reshape(-1))
        for i, [name, xyxy] in enumerate(zip(names,xy.int())):
            #print(f'Элемент №{i}\nКласс {name}\nКоординаты {[x,y,x1,y1]}',end='\n\n')
            x,y,x1,y1 = [int(i) for i in xyxy]
            cv.rectangle(image,(x,y),(x1,y1),(255,0,0),2)
            cv.putText(image, name, (x, y+10), 
                       cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
        #for x,y in xy.reshape(-1,2):                   
        #    cv.circle(frame2, (int(x),int(y)),3,(255,0,0),-1) 
        # #rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
cv.imshow('img',image)
if cv.waitKey(0) != -1: 
    cv.destroyAllWindows()

In [ ]:
'''
import subprocess
from subprocess import call

a = call('tg.bat')
if a == 0: print('Успешно!')
else: print('Не выполнено(')
'''
    
'''  
while True:
    volume = get_system_volume()
    a = int(input())
    if a == 0: UP = False
    if a == 1: UP = True
    if a == 2: break
    volume = volume + 0.1 if UP else volume - 0.1
    volume = 1 if volume > 1 else volume
    volume = 0 if volume < 0 else volume

    set_system_volume(volume)

    print(f"Текущая громкость: {volume}")
'''


'  \nwhile True:\n    volume = get_system_volume()\n    a = int(input())\n    if a == 0: UP = False\n    if a == 1: UP = True\n    if a == 2: break\n    volume = volume + 0.1 if UP else volume - 0.1\n    volume = 1 if volume > 1 else volume\n    volume = 0 if volume < 0 else volume\n\n    set_system_volume(volume)\n\n    print(f"Текущая громкость: {volume}")\n'

In [4]:
#Елка без обучения

cap = cv.VideoCapture(0)

if not cap.isOpened():
    print('Камера недоступна')
    
time_t = time.time()

model = YOLO("yolo11n-pose.pt")
_ , frame2 = cap.read()

while cap.isOpened():
    ret, frame = cap.read()

    frame = cv.resize(frame, (448, 448))

    #frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    if cv.waitKey(1) & 0xFF == ord('q'):
        break

    if not ret:
        break
      


    if cv.waitKey(1) & 0xFF == ord('p'):
        cv.imwrite('hand.png', frame)
    
    if time.time() - time_t >= 5:
        time_t = time.time()
        frame2 = frame.copy()
        results = model(frame)
        for result in results:
            xy = result.keypoints.xy
            for x,y in xy.reshape(-1,2):
                
                cv.circle(frame2, (int(x),int(y)),3,(255,0,0),-1) #rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        
    cv.imshow('frame2', frame2)


    cv.imshow('frame', frame)

cap.release()
            
cv.destroyAllWindows()


0: 640x640 1 person, 189.7ms
Speed: 6.2ms preprocess, 189.7ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 218.3ms
Speed: 5.6ms preprocess, 218.3ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 187.7ms
Speed: 8.6ms preprocess, 187.7ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 225.6ms
Speed: 5.5ms preprocess, 225.6ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 198.4ms
Speed: 9.1ms preprocess, 198.4ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 194.6ms
Speed: 5.1ms preprocess, 194.6ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 320.4ms
Speed: 7.6ms preprocess, 320.4ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 190.9ms
Speed: 8.2ms preprocess, 190.9ms inference, 1.7ms postprocess per image at

In [5]:
for x,y in xy.reshape(-1,2):
    print(int(x),y)

216 tensor(256.8307)
214 tensor(233.0921)
183 tensor(244.4566)
214 tensor(235.9136)
133 tensor(268.2428)
250 tensor(334.7099)
133 tensor(380.6685)
348 tensor(396.6926)
151 tensor(448.)
432 tensor(406.7729)
277 tensor(448.)
319 tensor(429.2288)
249 tensor(448.)
371 tensor(407.1001)
318 tensor(448.)
388 tensor(386.6938)
365 tensor(444.7682)


In [15]:
xy.reshape(-1,2)

tensor([[125.9410, 102.5305],
        [129.6494,  93.8843],
        [115.7516,  89.9513],
        [128.3413, 100.1764],
        [ 87.5500,  92.2971],
        [140.1980, 155.5047],
        [ 64.5515, 159.6040],
        [186.1358, 204.3116],
        [ 65.7252, 240.9517],
        [155.4243, 141.9862],
        [134.2363, 209.3454],
        [151.8249, 246.5921],
        [103.7287, 250.0000],
        [159.7180, 225.7550],
        [131.9483, 223.7973],
        [160.0715, 236.0619],
        [149.8042, 222.4153]])

In [6]:
model = YOLO("yolo11n.pt")

In [10]:
#Понять движение руки
#Теоретически, много слабых точек показывают движение ладонью
#Надо тестить дальше

cap = cv.VideoCapture(0)

if not cap.isOpened():
    print('Камера недоступна')
    
time_t = time.time()

model = YOLO("yolo11n-pose.pt")
_ , frame2 = cap.read()
frame2 = cv.resize(frame2, (448, 448))
frame2 = cv.cvtColor(frame2, cv.COLOR_BGR2GRAY)

orb = cv.ORB_create()
sift = cv.SIFT_create()
akaze = cv.AKAZE_create()

detectors = {
    'SIFT': cv.SIFT_create(),
    'ORB': cv.ORB_create(),
    'AKAZE': cv.AKAZE_create()
}


while cap.isOpened():
    ret, frame1 = cap.read()
    

    frame = cv.resize(frame1, (448, 448))

    frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    
    if cv.waitKey(1) & 0xFF == ord('q'):
        break

    if not ret:
        break

    #blur = cv.GaussianBlur(nor, (5,5), 0)    

    #ret, frame2 = cv.threshold(blur,0,255,cv.THRESH_BINARY+cv.THRESH_OTSU)
    
    kp, des = orb.detectAndCompute(frame, None)
    kp2 = kp
    kp2 = [k for k in kp if k.response < 0.000025]
    kp_len = len(kp2)
    '''
    for name, detector in detectors.items():
        kp, des = detector.detectAndCompute(frame, None)
        kp2 = [k for k in kp if k.response < 0.000015]
        frame2 = cv.drawKeypoints(frame, kp2, None, color=(0,255,0), 
                              flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
        cv.putText(frame2,str(len(kp2)),(10,50), cv.FONT_HERSHEY_SIMPLEX, 2,(255,255,0),2,cv.LINE_AA)
        cv.imshow(name, frame2)
        #print(f'Детектор: {name}; длина {len(kp2)}')

    '''
    weak_kp = sorted(kp2, key=lambda x: x.response, reverse=False)
    frame2 = cv.drawKeypoints(frame, weak_kp, None, color=(0,255,0), 
                              flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    check = ''
    if (kp_len > 10): 
        check = 'CHECK'
        #print('CHECK', kp_len)

    if cv.waitKey(1) & 0xFF == ord('p'):
        cv.imwrite('hand.png', frame)
    
    if time.time() - time_t >= 5:
        time_t = time.time()  



    cv.putText(frame2,str(len(weak_kp)) + check,(10,50), cv.FONT_HERSHEY_SIMPLEX, 2,(255,255,0),2,cv.LINE_AA)

    

    cv.imshow('frame', frame1)
    cv.imshow('frame2', frame2)

cap.release()
            
cv.destroyAllWindows()

In [27]:
strong_kp = sorted(kp2, key=lambda x: x.response, reverse=False)[:50]

In [28]:
for k in [k for k in strong_kp]:
    print(f'{k.pt[0]},{k.pt[1]} = size {k.size}, angle {k.angle} response {k.response}')

326.3832702636719,286.51727294921875 = size 11.416387557983398, angle 356.6027526855469 response 0.001030334853567183
239.27096557617188,285.5719299316406 = size 5.708193778991699, angle 177.88629150390625 response 0.0010312104132026434
372.4997253417969,364.4198913574219 = size 5.708193778991699, angle 276.8445739746094 response 0.0010351256933063269
247.2702178955078,273.7908630371094 = size 11.416387557983398, angle 175.07972717285156 response 0.0010388788068667054
101.85086059570312,326.64776611328125 = size 11.416387557983398, angle 338.1043395996094 response 0.0010448633693158627
312.7417907714844,157.7180938720703 = size 11.416387557983398, angle 343.1418762207031 response 0.0010493178851902485
209.07582092285156,51.564964294433594 = size 4.800000190734863, angle 151.48794555664062 response 0.0010561624076217413
304.2384338378906,79.2389907836914 = size 8.072606086730957, angle 333.2278747558594 response 0.0010716002434492111
136.91549682617188,104.22086334228516 = size 5.708193

In [20]:
for k in [k for k in kp if k.size > 110]:
    print(f'{k.pt[0]},{k.pt[1]} = size {k.size}, angle {k.angle} response {k.response}')

In [46]:
max([k.size for k in kp])

111.0786361694336

In [14]:
0.000025 == 25e-6

True